# Workshop Setup: Databricks Environment

This notebook provisions everything the labs need on the Databricks side. Run it top to bottom as an admin preparing a shared workspace, or as a participant self-serving in your own workspace or Free Edition account.

**The dual-database architecture.** The workshop dataset is an aircraft digital twin split across two databases. Neo4j holds the relationship-rich data: aircraft topology, flights, delays, and maintenance events. Databricks Delta tables hold the high-volume sensor telemetry. Aircraft, Systems, and Sensors exist in both databases and act as join points between them.

**Where this fits in the labs:**

- **Lab 1**: Neo4j Aura setup and Cypher introduction
- **Lab 2**: ETL from the lakehouse into Neo4j with the Spark Connector
- **Lab 3**: GraphRAG semantic search over maintenance manuals
- **Lab 4**: Genie space and Supervisor Agent
- **Lab 5**: Aura Agents: Create with AI

**Before you run this, you need:**

- Access to a Databricks workspace. Free Edition works for Steps 2 through 5; see the classic compute note in Step 1.
- A Unity Catalog catalog, created before you run Step 2, or the name of a shared catalog assigned by your instructor. Step 2 lists the default name.
- A running SQL warehouse (Free Edition ships with a Starter Warehouse).
- A Neo4j Aura instance with its URI, username, and password for the later labs. You create it in Lab 1, so it is fine to run this notebook first.

**What this notebook sets up:**

1. A classic compute cluster with the Neo4j Spark Connector and Python libraries, created in the UI or by the optional automation cell, plus a Neo4j connectivity test.
2. A Unity Catalog schema and volume inside the catalog you created.
3. The workshop data, downloaded from GitHub into the volume.
4. Four Delta lakehouse tables with Genie-friendly comments, plus a row-count verification.

## Step 1: Classic compute cluster and libraries

Labs 2 and 3 need a classic cluster because the Neo4j Spark Connector is a Maven library, which serverless compute cannot install. Create the cluster one of two ways.

**Option A: create the cluster manually.** In the UI (**Compute > Create compute**), configure:

- **Single node**, **Dedicated (single user)** access mode.
- Runtime **17.3.x-cpu-ml-scala2.13** (17.3 LTS ML on Spark 4.0).
- Node type **m5.large** on AWS, or the equivalent 2-core, 8 GB type on Azure or GCP.
- **Auto-termination: 30 minutes.**

Then install these libraries on the cluster (**Libraries > Install new**):

- **Maven**: `org.neo4j:neo4j-connector-apache-spark_2.13:5.4.3_for_spark_3`. This is the latest `for_spark_3` build of the connector and has been extensively tested to work with Spark 4.0 on Databricks Runtime 17.3 LTS ML.
- **PyPI**: `neo4j==6.2.0`, `databricks-agents>=1.11.0`, `langgraph==1.2.4`, `langchain-openai==1.3.0`, `pydantic==2.13.4`, `langchain-core>=1.4.6`, `databricks-langchain>=0.20.0`, `dspy>=3.2.1`, `neo4j-graphrag>=1.17.0`, `beautifulsoup4>=4.15.0`, `sentence_transformers`

**Option B: create the cluster automatically.** The next cell calls `create_workshop_cluster()` from `cluster_setup.py`, a Python file alongside this notebook. The function uses the Databricks SDK to create the cluster with the configuration above, start it, and install all the libraries. It requires cluster-create permission. Skip the cell if you used Option A.

**Free Edition note**: Free Edition provides serverless compute only and cannot create classic clusters, so this step needs a standard workspace. The rest of this notebook (Steps 2 through 5) runs fine on Free Edition; ask your instructor how Labs 2 and 3 are run in your session.

In [ ]:
# OPTIONAL: skip this cell if you created the cluster manually (Option A above).
from cluster_setup import create_workshop_cluster

cluster_id = create_workshop_cluster()
print(f"Cluster {cluster_id} is running with all libraries installed.")

### Test the cluster with a basic Neo4j Spark job

This read-only test proves the Spark Connector is installed and can reach your Neo4j instance. Two prerequisites:

- **Attach this notebook to the new classic cluster.** The connector is a cluster library, so this cell fails on serverless compute.
- **Enter your Neo4j credentials in the configuration block.** They come from the Aura instance you create in Lab 1. If you have not done Lab 1 yet, skip this cell and come back after.

The node count returns 0 on a fresh Aura instance. That still proves connectivity; the data arrives in Lab 2.

In [ ]:
# ============================================
# CONFIGURATION - Enter your Neo4j credentials
# ============================================

NEO4J_URI = ""  # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""  # Your password from Lab 1

if not NEO4J_URI or not NEO4J_PASSWORD:
    raise ValueError("Enter your Neo4j credentials above before running this cell.")

spark.conf.set("neo4j.url", NEO4J_URI)
spark.conf.set("neo4j.authentication.basic.username", NEO4J_USERNAME)
spark.conf.set("neo4j.authentication.basic.password", NEO4J_PASSWORD)
spark.conf.set("neo4j.database", "neo4j")

test_df = (
    spark.read.format("org.neo4j.spark.DataSource")
    .option("query", "RETURN 1 AS connection_test")
    .load()
)
display(test_df)

count_df = (
    spark.read.format("org.neo4j.spark.DataSource")
    .option("query", "MATCH (n) RETURN count(n) AS node_count")
    .load()
)
display(count_df)
print("Neo4j Spark Connector is working.")

## Step 2: Catalog, schema, and volume

The workshop uses one Unity Catalog catalog, one schema, and one volume. The schema holds both the volume of raw CSVs and manuals and the Delta tables created in Step 4. The defaults are:

`databricks-neo4j-workshop`.`aircraft`.`raw_data`

**Create the catalog first.** The next cell does not create it; it stops with an error if the catalog is missing. In the UI, go to **Catalog > Create catalog** and create `databricks-neo4j-workshop`, or set the `CATALOG` constant to a catalog an admin created for you. The cell then creates the schema and volume inside it. Two notes:

- **Shared workspace, participant**: prefix the catalog with your username, for example `jdoe_databricks-neo4j-workshop`, so you do not collide with other participants.
- Catalog creation needs metastore privileges. If you cannot create one, ask your instructor for the name of a shared catalog and set `CATALOG` to it.

If you change a constant later, re-run the notebook from the next cell down so the new names take effect.

In [ ]:
CATALOG = "databricks-neo4j-workshop"
SCHEMA = "aircraft"
VOLUME = "raw_data"

volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
target = f"`{CATALOG}`.`{SCHEMA}`"  # used by the lakehouse cells below

if spark.sql(f"SHOW CATALOGS LIKE '{CATALOG}'").count() == 0:
    raise RuntimeError(
        f"Catalog `{CATALOG}` not found. Create it manually first (Catalog > "
        "Create catalog in the UI), or set CATALOG to a catalog an admin created "
        "for you, then re-run this cell."
    )
print(f"Catalog `{CATALOG}` found")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {target}.`{VOLUME}`")
print(f"Volume ready at {volume_path}")

## Step 3: Download the workshop data from GitHub

The dataset (22 CSVs plus 5 maintenance manuals, about 35 MB) is committed to the public workshop repo, pinned to a release tag so the data cannot change mid-class. This cell downloads each file straight from `raw.githubusercontent.com` into the volume and skips files that are already there, so re-runs are cheap.

If your workspace blocks outbound internet access, clone the repo as a Git folder instead and copy the files from `workshop-setup/aircraft_digital_twin_data/` into the volume.

In [ ]:
import urllib.error
import urllib.request
from pathlib import Path

DATA_TAG = "v1.1.0"
GITHUB_DATA_BASE = (
    "https://raw.githubusercontent.com/neo4j-partners/databricks-neo4j-workshop/"
    f"{DATA_TAG}/workshop-setup/aircraft_digital_twin_data"
)

MAINTENANCE_MANUALS = [
    "MAINTENANCE_A220.md",
    "MAINTENANCE_A320.md",
    "MAINTENANCE_A321neo.md",
    "MAINTENANCE_B737.md",
    "MAINTENANCE_E190.md",
]
NODE_CSVS = [
    "nodes_aircraft.csv",
    "nodes_airports.csv",
    "nodes_components.csv",
    "nodes_delays.csv",
    "nodes_flights.csv",
    "nodes_maintenance.csv",
    "nodes_readings.csv",
    "nodes_removals.csv",
    "nodes_sensors.csv",
    "nodes_systems.csv",
]
REL_CSVS = [
    "rels_aircraft_flight.csv",
    "rels_aircraft_removal.csv",
    "rels_aircraft_system.csv",
    "rels_component_event.csv",
    "rels_component_removal.csv",
    "rels_event_aircraft.csv",
    "rels_event_system.csv",
    "rels_flight_arrival.csv",
    "rels_flight_delay.csv",
    "rels_flight_departure.csv",
    "rels_system_component.csv",
    "rels_system_sensor.csv",
]
DATA_FILES = MAINTENANCE_MANUALS + NODE_CSVS + REL_CSVS


def download(url: str, dest: Path) -> None:
    """Download url to dest, only moving it into place once fully written."""
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            data = response.read()
            expected = int(response.headers.get("Content-Length", len(data)))
    except urllib.error.HTTPError as err:
        raise RuntimeError(
            f"Download failed ({err.code}) for {url}. If this is a 404, check that "
            f"the release tag {DATA_TAG} exists on the repo."
        ) from err
    if len(data) != expected:
        raise RuntimeError(f"Incomplete download of {url}: got {len(data)} of {expected} bytes")
    tmp = dest.with_name(dest.name + ".tmp")
    tmp.write_bytes(data)
    tmp.rename(dest)


volume_dir = Path(volume_path)
downloaded, skipped = 0, 0
for name in DATA_FILES:
    dest = volume_dir / name
    if dest.exists() and dest.stat().st_size > 0:
        skipped += 1
        continue
    download(f"{GITHUB_DATA_BASE}/{name}", dest)
    print(f"  downloaded {name}")
    downloaded += 1
print(f"{downloaded} downloaded, {skipped} already in the volume")

present = {f.name for f in dbutils.fs.ls(volume_path)}
missing = [name for name in DATA_FILES if name not in present]
if missing:
    raise RuntimeError(f"Files missing from the volume: {missing}")
print(f"All {len(DATA_FILES)} workshop files are in {volume_path}")
display(dbutils.fs.ls(volume_path))

## Step 4: Lakehouse tables

Create the four Delta tables that Genie and the labs query, straight from the CSVs in the volume. Each table reads its specific file, never a glob, because the volume also holds the other node and relationship CSVs that only Neo4j uses. The `sensor_readings` table is partitioned by `sensor_id` since the labs filter by sensor.

In [ ]:
# Uses `target` and `volume_path` from Step 2; run those cells first after a restart.
TBLPROPS = "TBLPROPERTIES ('delta.columnMapping.mode' = 'name')"


def read_csv_expr(filename: str) -> str:
    """Return a read_files() expression for one CSV in the volume."""
    return (
        f"read_files('{volume_path}/{filename}', "
        "format => 'csv', header => 'true', inferSchema => 'true')"
    )


steps = [
    (
        "Creating aircraft table",
        f"""CREATE TABLE IF NOT EXISTS {target}.aircraft
        {TBLPROPS}
        AS SELECT * FROM {read_csv_expr("nodes_aircraft.csv")}""",
    ),
    (
        "Creating systems table",
        f"""CREATE TABLE IF NOT EXISTS {target}.systems
        {TBLPROPS}
        AS SELECT * FROM {read_csv_expr("nodes_systems.csv")}""",
    ),
    (
        "Creating sensors table",
        f"""CREATE TABLE IF NOT EXISTS {target}.sensors
        {TBLPROPS}
        AS SELECT * FROM {read_csv_expr("nodes_sensors.csv")}""",
    ),
    (
        "Creating sensor_readings table",
        f"""CREATE TABLE IF NOT EXISTS {target}.sensor_readings
        {TBLPROPS}
        PARTITIONED BY (sensor_id)
        AS SELECT
            reading_id,
            sensor_id,
            to_timestamp(ts) AS timestamp,
            CAST(value AS DOUBLE) AS value
        FROM {read_csv_expr("nodes_readings.csv")}""",
    ),
]

for description, sql in steps:
    print(f"{description}...")
    spark.sql(sql)
print("Tables created.")

### Genie table and column comments

Table and column comments teach Lab 4's Genie space what the data model means. Without them, natural language questions like "which sensor had the highest exhaust gas temperature" have nothing to ground against.

In [ ]:
# Uses `target` from Step 2; run those cells first after a restart.
COMMENTS = [
    # Aircraft table
    f"COMMENT ON TABLE {target}.aircraft IS 'Fleet of aircraft with tail numbers, models, and operators'",
    f"COMMENT ON COLUMN {target}.aircraft.`:ID(Aircraft)` IS 'Unique aircraft identifier'",
    f"COMMENT ON COLUMN {target}.aircraft.tail_number IS 'Aircraft registration/tail number (e.g., N95040A)'",
    f"COMMENT ON COLUMN {target}.aircraft.model IS 'Aircraft model (e.g., B737-800, A320-200)'",
    f"COMMENT ON COLUMN {target}.aircraft.operator IS 'Airline operator name'",
    # Systems table
    f"COMMENT ON TABLE {target}.systems IS 'Aircraft systems including engines, avionics, and hydraulics'",
    f"COMMENT ON COLUMN {target}.systems.`:ID(System)` IS 'Unique system identifier'",
    f"COMMENT ON COLUMN {target}.systems.type IS 'System type (Engine, Avionics, Hydraulics)'",
    f"COMMENT ON COLUMN {target}.systems.name IS 'Human-readable system name'",
    # Sensors table
    f"COMMENT ON TABLE {target}.sensors IS 'Sensors installed on aircraft systems'",
    f"COMMENT ON COLUMN {target}.sensors.`:ID(Sensor)` IS 'Unique sensor identifier'",
    f"COMMENT ON COLUMN {target}.sensors.type IS 'Sensor type: EGT (Exhaust Gas Temperature in Celsius), Vibration (ips), N1Speed (RPM), FuelFlow (kg/s)'",
    f"COMMENT ON COLUMN {target}.sensors.unit IS 'Unit of measurement'",
    # Sensor readings table
    f"COMMENT ON TABLE {target}.sensor_readings IS 'Sensor readings every 4 hours over 90 days (July-September 2024)'",
    f"COMMENT ON COLUMN {target}.sensor_readings.reading_id IS 'Unique reading identifier'",
    f"COMMENT ON COLUMN {target}.sensor_readings.sensor_id IS 'Foreign key to sensors table'",
    f"COMMENT ON COLUMN {target}.sensor_readings.timestamp IS 'Reading timestamp (4-hour intervals)'",
    f"COMMENT ON COLUMN {target}.sensor_readings.value IS 'Sensor reading value in the sensor unit'",
]

for statement in COMMENTS:
    spark.sql(statement)
print(f"Applied {len(COMMENTS)} table and column comments")

### Verify row counts

The expected counts match the dataset pinned at the release tag: 100 aircraft, 400 systems, 800 sensors, and 432,000 readings (800 sensors, every 4 hours, 90 days).

In [ ]:
# Uses `target` from Step 2; run those cells first after a restart.
EXPECTED_ROW_COUNTS = {
    "aircraft": 100,
    "systems": 400,
    "sensors": 800,
    "sensor_readings": 432_000,
}

counts = spark.sql(f"""
    SELECT 'aircraft' AS table_name, COUNT(*) AS row_count FROM {target}.aircraft
    UNION ALL
    SELECT 'systems', COUNT(*) FROM {target}.systems
    UNION ALL
    SELECT 'sensors', COUNT(*) FROM {target}.sensors
    UNION ALL
    SELECT 'sensor_readings', COUNT(*) FROM {target}.sensor_readings
""")
display(counts)

actual = {row["table_name"]: row["row_count"] for row in counts.collect()}
mismatches = []
for table, expected in EXPECTED_ROW_COUNTS.items():
    count = actual.get(table, 0)
    if count != expected:
        mismatches.append(f"{table}: got {count}, expected {expected}")

if mismatches:
    raise RuntimeError("Row counts do not match the workshop dataset: " + "; ".join(mismatches))
print("All row counts match. Databricks setup is complete.")

## Step 5: Check for a SQL warehouse

The labs and Genie need a running SQL warehouse. Free Edition and most workspaces ship with a Starter Warehouse, so this cell only checks that one exists; it does not create one.

In [ ]:
from databricks.sdk import WorkspaceClient

client = WorkspaceClient()
warehouses = list(client.warehouses.list())
if warehouses:
    for warehouse in warehouses:
        print(f"  {warehouse.name}: {warehouse.state}")
else:
    print("No SQL warehouse found. Labs 2-4 and Genie need one; ask your admin to")
    print("create a warehouse, or use the Starter Warehouse on Free Edition.")

## What this notebook does not cover

- **Neo4j Aura instance**: you create a free Aura instance in Lab 1 and record its URI, username, and password. Enter them in the configuration cells of the Lab 2 and Lab 3 notebooks, or store them in Databricks secrets.
- **Neo4j data load**: the graph side is populated by the Lab 2 ETL notebooks from the volume, or by the admin CLI `populate_aircraft_db` for the GraphRAG enrichment used in Lab 3.
- **Lab notebooks in the workspace**: `databricks-setup sync` uploads the Lab 2, Lab 3, and MCP notebooks to `/Shared/databricks-neo4j-workshop`. Cloning the repo as a Git folder works too.
- **Unity Catalog grants (shared workspace admins)**: grant participants `USE CATALOG`, `USE SCHEMA`, `READ VOLUME`, and `SELECT` on the lakehouse tables.
- **Lab 4 components**: the Genie space, the Unity Catalog HTTP connection to the Neo4j MCP server, and the Supervisor Agent in Agent Bricks are all created in Lab 4 itself.
- **Verification of the graph side**: run `verify-labs` after Lab 2; the row-count cell above already covers the Databricks side.